# מפגש 22 — Pipeline ו־Cross Validation | מחברת מורה Audited
**90 דקות הוראה ותרגול + 45 דקות פרויקט**. בהשוואת מודלים תלמידים לא אמורים לראות מראש את שאלות המבחן. נריץ חמש חלוקות אימות פנימיות, בכל פעם נאמן מחדש את Pipeline על ארבעת החלקים האחרים. ההדגמה משתמשת ב־flipper_length_mm וב־body_mass_g, seed=42. לעומת זאת מעבדת התלמיד משתמשת ב־bill_length_mm ו־bill_depth_mm עם seed=17.

עצרו לפני כל תא: חזוי → הרצה → פירוש → מגבלה. הקובץ דורש גישה ל־CSV של הקורס.

## א. חישוב ידני של ממוצע קיפולים (סינתטי)
שלושה ציונים מומצאים: 0.80, 0.90, 0.70. חשבו ידנית (0.8+0.9+0.7)/3. זהו ממוצע אימות, **לא Test חיצוני**, והנתונים אינם תוצאות Penguins.

In [ ]:
import numpy as np
scores=np.array([.8,.9,.7])
assert len(scores)==3
assert np.isclose(scores.mean(),.8)
assert np.isclose(scores.sum(),2.4)
print("Synthetic CV mean:",scores.mean())

## ב. הכנת פיתוח וחלוקה חיצונית
Train מכיל את הדוגמאות מהן לומדים או בוחרים; Test מוחזק בצד. הקפידו על stratify ועל seed. אין לבצע Imputer.fit או Scaler.fit על כל Train לפני CV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
assert {"species","flipper_length_mm","body_mass_g","bill_length_mm","bill_depth_mm"}.issubset(df.columns)
assert df.shape[0]==344
features=["flipper_length_mm","body_mass_g"]
data=df.dropna(subset=["species"]).copy()
X=data[features].copy()
y=data["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
assert len(X_train)==258
assert len(X_test)==86
assert X_train.index.is_unique
assert X_test.index.is_unique
assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train)+len(X_test)==len(y)
assert X_train.columns.tolist()==features
print("Outer split:",X_train.shape,X_test.shape)

## ג. חמשת קיפולי האימות
בכל קיפול חלק אחר של Outer Train הוא Validation. עקבו אחר היעדר חפיפה ואחר העובדה שכל רשומה מתפקדת פעם אחת כאימות. השימוש באותם קיפולים לכל K מאפשר השוואה הוגנת.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
seen=np.zeros(len(X_train),dtype=int)
for fold_number,(inner_train,inner_validation) in enumerate(cv.split(X_train,y_train),start=1):
    assert set(inner_train).isdisjoint(inner_validation)
    assert len(inner_train)+len(inner_validation)==len(X_train)
    assert set(inner_train).union(inner_validation)==set(range(len(X_train)))
    assert set(y_train.iloc[inner_validation])==set(y_train.unique())
    seen[inner_validation]+=1
    print("Fold",fold_number,"train",len(inner_train),"validation",len(inner_validation))
assert np.all(seen==1)
print("All outer Train rows validated once inside CV")

## ד. מה עלול לדלוף? ניסוי קטן
השתמשו בממוצע 0,2,4 של האימון הפנימי בלבד, לעומת ממוצע שכולל בטעות את 100 מרשומת האימות. למה זה פסול גם אם ציון המדד נראה טוב?

In [ ]:
# ניסוי סינתטי להמחשת מניעת זליגת מידע; אינו חלק מציוני Penguins
tiny_train=np.array([[0.0],[2.0],[4.0]])
tiny_validation=np.array([[100.0]])
safe_scaler=StandardScaler().fit(tiny_train)
unsafe_scaler=StandardScaler().fit(np.vstack([tiny_train,tiny_validation]))
assert np.isclose(safe_scaler.mean_[0],2.0)
assert np.isclose(unsafe_scaler.mean_[0],26.5)
assert not np.isclose(safe_scaler.mean_[0],unsafe_scaler.mean_[0])
assert np.isclose(safe_scaler.transform(tiny_validation)[0,0],(100-2)/safe_scaler.scale_[0])
print("Training-only mean:",safe_scaler.mean_[0],"leaked mean:",unsafe_scaler.mean_[0])

## ה. Pipeline של השלמה → תקנון → KNN
ההרצה אינה מאמנת על Test. scikit-learn מתאים מחדש כל רכיב בתוך cross_validate. השדה test_score בתוך cross_validate הוא **inner Validation score**.

In [ ]:
candidates=[1,3,5,15,51]
metric="f1_macro"
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
fold_detail={}
for k in candidates:
    pipeline=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),
                           KNeighborsClassifier(n_neighbors=k,weights="uniform",metric="euclidean"))
    fold=cross_validate(pipeline,X_train,y_train,cv=cv,scoring=metric,
                        return_train_score=True,n_jobs=1)
    assert len(fold["test_score"])==5
    assert len(fold["train_score"])==5
    assert np.isfinite(fold["test_score"]).all()
    fold_detail[k]=fold["test_score"].copy()
    rows.append({"K":k,"train_mean":fold["train_score"].mean(),
                 "cv_mean":fold["test_score"].mean(),
                 "cv_std":fold["test_score"].std(ddof=0)})
results_df=pd.DataFrame(rows)
assert results_df["K"].tolist()==candidates
assert len(results_df)==5
assert results_df["cv_mean"].between(0,1).all()
assert results_df["cv_std"].ge(0).all()
assert all(len(scores)==5 for scores in fold_detail.values())
display(results_df)

## ו. פירוש מדדים
`train_mean` מתייחס לקיפולי האימון, `cv_mean` לקיפולי האימות, ו־`cv_std` לפיזור הקיפולים. סטיית התקן אינה רווח סמך; פער קטן אינו הוכחת איכות.

In [ ]:
assert set(["K","train_mean","cv_mean","cv_std"]).issubset(results_df.columns)
assert (results_df["train_mean"]>=0).all()
assert (results_df["train_mean"]<=1).all()
ax=results_df.plot(x="K",y=["train_mean","cv_mean"],marker="o",title="Outer Train only: inner CV scores")
ax.set_ylabel("Macro-F1")
plt.show()

## ז. בחירת K ואז התאמה מחדש
הכלל קבוע מראש: cv_mean הגבוה ביותר, ובשוויון מדויק K הקטן. יוצרים Pipeline חדשה ומתאימים לכל Outer Train. רק אחרי הבחירה בודקים Test פעם אחת.

In [ ]:
ranked=results_df.sort_values(["cv_mean","K"],ascending=[False,True])
best_k=int(ranked.iloc[0]["K"])
assert best_k in candidates
assert np.isclose(ranked.iloc[0]["cv_mean"],results_df["cv_mean"].max())
assert results_df.loc[results_df["K"]==best_k,"cv_mean"].size==1
final_model=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),
                          KNeighborsClassifier(n_neighbors=best_k,weights="uniform",metric="euclidean"))
final_model.fit(X_train,y_train)
assert final_model.named_steps["kneighborsclassifier"].n_neighbors==best_k
assert final_model.named_steps["standardscaler"].n_features_in_==2
predictions=final_model.predict(X_test)
assert len(predictions)==len(y_test)==86
test_f1=f1_score(y_test,predictions,average="macro")
test_acc=accuracy_score(y_test,predictions)
assert 0<=test_f1<=1
assert 0<=test_acc<=1
report=classification_report(y_test,predictions,labels=["Adelie","Chinstrap","Gentoo"],output_dict=True,zero_division=0)
assert sum(report[x]["support"] for x in ["Adelie","Chinstrap","Gentoo"])==86
print("Chosen by inner CV:",best_k,"Outer Test macro-F1:",test_f1,"Accuracy:",test_acc)

## כרטיס יציאה ופרויקט
1. מה דולף כאשר מתקננים לפני CV? 2. למה cross_validate.test_score אינו ציון Test החיצוני? 3. מדוע seed חדש אינו מוחק חשיפה קודמת? 4. אם יש כמה רשומות לאותו אדם, מה צריך לשנות בתכנון החלוקה? השאירו 45 דקות לפרויקט.